In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

In [ ]:
customers = pd.read_csv('../data/raw/customers.csv')
sessions = pd.read_csv('../data/raw/sessions.csv')
cart = pd.read_csv('../data/raw/cart_events.csv')
orders = pd.read_csv('../data/raw/orders.csv')
products = pd.read_csv('../data/raw/products.csv')

In [ ]:
def validate(df, name):
    print(f"\n📊 {name}")
    summary = pd.DataFrame({
        "Data Type": df.dtypes,
        "Missing Values": df.isnull().sum(),
        "Unique Values": df.nunique()
    })
    print(summary)
    print("Shape:", df.shape)
    print("Duplicates:", df.duplicated().sum())


#validating each dataframe
validate(customers, "Customers")
validate(sessions, "Sessions")
validate(cart, "Cart")
validate(orders, "Orders")
validate(products, "Products")

In [ ]:


# --- Customers table ---
# Convert signup_date from object (string) to datetime
customers['signup_date'] = pd.to_datetime(customers['signup_date'])

# --- Sessions table ---
# Convert session_date from object to datetime and session_duration from seconds (int) to timedelta
sessions['session_date'] = pd.to_datetime(sessions['session_date'])
sessions['session_duration'] = pd.to_timedelta(sessions['session_duration'], unit='s')


# --- Cart table ---
# Convert event_timestamp from object to datetime
cart['event_timestamp'] = pd.to_datetime(cart['event_timestamp'])

# --- Orders table ---
# Convert order_date from object to datetime
orders['order_date'] = pd.to_datetime(orders['order_date'])

#converting categorical columns to category dtype for memory efficiency
sessions['bounced_flag'] = sessions['bounced_flag'].astype('category')
orders['ab_group'] = orders['ab_group'].astype('category')
products['category'] = products['category'].astype('category')

In [ ]:
### KPI Definitions 

| KPI Name              | Formula                                | Description |
|-----------------------|----------------------------------------|-------------|
| Total Users           | Count of unique customer_id            | Total number of distinct customers/users on the platform |
| Total Sessions        | Count of session_id                    | Total number of visits or browsing sessions |
| Total Orders          | Count of order_id                      | Total number of completed purchase transactions |
| Total Revenue         | Sum(order_value)                       | Total monetary value of all completed orders |
| Conversion Rate       | (Total Orders ÷ Total Sessions) × 100  | Percentage of sessions that result in a purchase |
| Add-to-Cart Rate      | (Cart Events ÷ Total Sessions) × 100   | Percentage of sessions where at least one item is added to cart |
| Cart Abandonment Rate | ((Cart Events − Orders) ÷ Cart Events) × 100 | Percentage of carts created that did not convert into orders |
| Average Order Value   | (Total Revenue ÷ Total Orders)         | Average revenue generated per order |
| Orders per Customer   | (Total Orders ÷ Total Users)           | Average number of orders placed per customer |


In [ ]:
total_users = customers['customer_id'].nunique()
total_sessions = sessions['session_id'].nunique()
total_orders = orders['order_id'].nunique()

orders['revenue'] = orders['final_price'] * orders['quantity']
total_revenue = orders['revenue'].sum()
total_margin = orders['margin'].sum()

conversion_rate = total_orders / total_sessions

total_cart = cart['cart_id'].nunique()
add_to_cart_rate = total_cart / total_sessions
cart_abandonment_rate = (total_cart - total_orders) / total_cart

average_order_value = total_revenue / total_orders
orders_per_customer = total_orders / total_users

# --- Print Results ---
print("Total Users:", total_users)
print("Total Sessions:", total_sessions)
print("Total Orders:", total_orders)
print("Total Revenue:", round(total_revenue, 2))
print("Conversion Rate:", round(conversion_rate, 2), "%")
print("Add-to-Cart Rate:", round(add_to_cart_rate, 2), "%")
print("Cart Abandonment Rate:", round(cart_abandonment_rate, 2), "%")
print("Average Order Value:", round(average_order_value, 2))
print("Orders per Customer:", round(orders_per_customer, 2))
